In [0]:
# ============================================================
# Gold Layer Setup
# ============================================================

CONFIG = {
    "silver_fact_table": "tvmaze.silver.fact_show_data",
    "gold_table": "tvmaze.gold.gold_metrics"
}

spark.sql("CREATE SCHEMA IF NOT EXISTS tvmaze.gold")

print("Gold schema ready")

In [0]:
# ============================================================
# Load Fact Table
# ============================================================

fact_df = spark.table(CONFIG["silver_fact_table"])

fact_df.printSchema()

print("Fact table loaded")

In [0]:
# ============================================================
# Episodes per Season
# ============================================================
from pyspark.sql.functions import count

episodes_per_season_df = (
    fact_df
    .groupBy(
        "show_id",
        "show_name",
        "season"
    )
    .agg(
        count("episode_name").alias("episodes_per_season")
    )
)

print("Episodes per season calculated")
# display(episodes_per_season_df)

In [0]:
# ============================================================
# Average Runtime per show
# ============================================================
from pyspark.sql.functions import avg

avg_runtime_df = (
    fact_df
    .groupBy(
        "show_id",
        "show_name"
    )
    .agg(
        avg("runtime").alias("avg_runtime_minutes")
    )
)

print("Average runtime calculated")
# display(avg_runtime_df)

In [0]:
# ============================================================
# Top Cast Members 
# ============================================================
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, count, col

cast_counts = (
    fact_df.groupBy("cast_name")
        .agg(count("*").alias("episode_appearances"))
)

w = Window.partitionBy().orderBy(col("episode_appearances").desc())

top_cast = (
    cast_counts
    .withColumn("rank", row_number().over(w))
)
# display(top_cast)

In [0]:
# ============================================================
# Popular Genre
# ============================================================
genres_df = (
    fact_df
    .groupBy("genre")
    .agg(
        count("*").alias("genre_frequency")
    ).orderBy(col('genre_frequency').desc())
)

print("Genre popularity calculated")
# display(genres_df)

In [0]:
# ============================================================
# Write Gold table
# ============================================================
gold_df = (
    episodes_per_season_df
    .join(
        avg_runtime_df,
        ["show_id", "show_name"],
        "left"
    )
)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(CONFIG["gold_table"])

print("Gold table created")

In [0]:
%sql
-- Databricks SQL query
SELECT 
    show_name,
    season,
    episodes_per_season,
    avg_runtime_minutes
FROM tvmaze.gold.gold_metrics
ORDER BY episodes_per_season DESC;

In [0]:
from datetime import datetime

log = {
    "notebook": "Data_Aggregations",
    "status": "Succeeded",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "notes": "Gold metrics table created"
}

print(log)
